# 00 — Comprendre les données et les profils clients

Une ligne est un **snapshot client × mois** ; un même client peut revenir plusieurs mois. `churn=1` désigne une résiliation dans les 30 jours suivants. Ce notebook décrit les données ; il n'entraîne aucun modèle et ne produit pas les features de modélisation.

Parcours : structure → panel et cohortes → distributions et relations → dix KPI descriptifs. Les commentaires posent une question et explicitent l'unité et les limites. L'exploration porte sur toute l'année : le test ultérieur n'est donc pas aveugle.

Matplotlib est affiché en SVG adaptable à la fenêtre. Les textes rétrécissent aussi ; ajuster `FIG_WIDTH` ou passer les grilles à une colonne pour une fenêtre étroite.

## 1 — Imports, affichage et lecture

La normalisation est identique au 01. Les copies ambiguës ne sont pas arbitrées. Les données brutes restent disponibles en mémoire pour audit.

In [1]:
import sys
from io import StringIO
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML, display

if not Path("settings.py").is_file() and Path("../settings.py").is_file():
    sys.path.insert(0, str(Path.cwd().parent))
from settings import settings

rendered_figures = {}
FIG_WIDTH = 11
PANEL_HEIGHT = 4
plt.rcParams.update({"axes.grid": True, "grid.alpha": 0.2, "font.size": 10})


def show_figure(fig, name):
    """Rendu vectoriel adaptable ; aucun fichier de figure hors MLflow."""
    fig.tight_layout()
    buffer = StringIO()
    fig.savefig(buffer, format="svg", bbox_inches="tight")
    svg = buffer.getvalue()
    rendered_figures[name] = svg
    svg = svg.replace("<svg ", '<svg style="width:100%;height:auto;display:block;" ', 1)
    display(HTML('<div style="max-width:100%;">' + svg + "</div>"))
    plt.close(fig)


def read_telco_source(path):
    """Normaliser une copie ; tracer les exclusions sans modifier le CSV source."""
    raw = pd.read_csv(path, dtype={"customer_id": "string", "dept": "string"})
    required = {
        "customer_id",
        "snapshot_date",
        "signup_date",
        "contract",
        "region",
        "dept",
        "channel",
        "monthly_fee",
        "data_gb",
        "support_calls_3m",
        "late_payments",
        "nps",
        "churn",
    }
    missing = required - set(raw.columns)
    if missing:
        raise ValueError(f"Colonnes absentes : {sorted(missing)}")
    data = raw.copy()
    for column in ["customer_id", "contract", "region", "dept", "channel"]:
        data[column] = data[column].astype("string").str.strip().str.strip("\"'")
    for column in ["contract", "region", "dept", "channel"]:
        data[column] = data[column].str.lower().replace("", pd.NA)
    for column in ["snapshot_date", "signup_date"]:
        data[column] = pd.to_datetime(data[column], errors="coerce")
    if (
        data.customer_id.isna().any()
        or data.customer_id.eq("").any()
        or data.snapshot_date.isna().any()
    ):
        raise ValueError("Identifiant ou snapshot_date manquant/invalide.")
    numeric = ["monthly_fee", "data_gb", "support_calls_3m", "late_payments", "nps", "churn"]
    numeric += [c for c in ["age", "last_invoice_amount", "retention_offer_declined"] if c in data]
    invalid_rows = []
    for column in numeric:
        values = pd.to_numeric(data[column], errors="coerce")
        invalid = data[column].notna() & values.isna()
        invalid |= values.notna() & ~np.isfinite(values)
        if column in [
            "monthly_fee",
            "data_gb",
            "support_calls_3m",
            "late_payments",
            "last_invoice_amount",
        ]:
            invalid |= values.lt(0)
        if column == "nps":
            invalid |= values.notna() & ~values.between(0, 10)
        if column == "churn":
            if invalid.any() or not values.dropna().isin([0, 1]).all():
                raise ValueError("La cible doit valoir 0, 1 ou être manquante.")
        data[column] = values.mask(invalid)
        invalid_rows.append({"column": column, "invalid_values": int(invalid.sum())})
    before = len(data)
    data = data.drop_duplicates().copy()
    copies_removed = before - len(data)
    ambiguous = data.duplicated(["customer_id", "snapshot_date"], keep=False)
    ambiguous_keys = data.loc[ambiguous, ["customer_id", "snapshot_date"]].drop_duplicates()
    data = data.loc[~ambiguous].copy()
    if data.empty:
        raise ValueError("Aucune observation non ambiguë.")
    if (
        data.assign(month=data.snapshot_date.dt.to_period("M"))
        .duplicated(["customer_id", "month"])
        .any()
    ):
        raise ValueError("Plusieurs snapshots dans le même mois pour un client.")
    quality = {
        "raw_rows": len(raw),
        "retained_rows": len(data),
        "normalized_copies_removed": copies_removed,
        "ambiguous_keys_excluded": len(ambiguous_keys),
        "missing_labels": int(data.churn.isna().sum()),
    }
    return raw, data, pd.DataFrame(invalid_rows), quality


raw_df, data, invalid_values, quality = read_telco_source(settings.DATA_PATH)
display(pd.Series(quality, name="value").to_frame(), invalid_values)

labeled = data.loc[data.churn.notna()].copy()
if labeled.empty:
    raise ValueError("Aucune cible disponible.")

,value
raw_rows,38689
retained_rows,38629
normalized_copies_removed,60
ambiguous_keys_excluded,0
missing_labels,0


,column,invalid_values
0,monthly_fee,113
1,data_gb,0
2,support_calls_3m,0
3,late_payments,0
4,nps,0
5,churn,0
6,age,0
7,last_invoice_amount,0
8,retention_offer_declined,0


## 2 — Composition des colonnes

Comparer les types bruts, les cardinalités et les valeurs manquantes. Les identifiants et départements restent du texte ; une absence de valeur ne signifie pas zéro.

In [2]:
profile = pd.DataFrame(
    {
        "column": data.columns,
        "raw_type": [str(raw_df[c].dtype) for c in data],
        "normalized_type": [str(data[c].dtype) for c in data],
        "distinct": [data[c].nunique() for c in data],
        "missing_pct": [data[c].isna().mean() * 100 for c in data],
    }
)
display(profile, data.head())
ordered = profile.sort_values("missing_pct", ascending=False)
fig, ax = plt.subplots(figsize=(FIG_WIDTH, 6))
ax.barh(ordered.column, ordered.missing_pct, color="#2563EB")
ax.invert_yaxis()
ax.set(title="Disponibilité des colonnes", xlabel="Snapshots manquants (%)")
show_figure(fig, "missingness")

,column,raw_type,normalized_type,distinct,missing_pct
0,customer_id,string,string,4000,0.000000
1,snapshot_date,object,datetime64[ns],12,0.000000
2,signup_date,object,datetime64[ns],1787,0.000000
3,age,float64,float64,72,5.501048
4,contract,object,string,3,0.000000
5,region,object,string,7,0.000000
6,dept,string,string,60,0.000000
7,channel,object,string,4,0.000000
8,monthly_fee,float64,float64,8162,5.200756
9,data_gb,float64,float64,508,0.000000


,customer_id,snapshot_date,signup_date,age,contract,region,dept,channel,monthly_fee,data_gb,support_calls_3m,late_payments,nps,last_invoice_amount,retention_offer_declined,churn
0,102683,2024-10-31,2020-12-04,38.0,biannuel,occ,02,partenaire,65.15,7.9,1,1,7.0,65.15,0,0
1,100127,2024-04-30,2020-05-25,40.0,mensuel,ara,01,telephone,48.21,14.2,0,0,NaN,48.21,0,0
2,102042,2024-04-30,2024-01-06,21.0,mensuel,idf,09,partenaire,49.09,5.8,1,0,10.0,49.09,0,0
3,101775,2024-02-29,2021-01-01,42.0,mensuel,occ,07,web,60.29,26.4,3,0,NaN,60.29,0,0
4,103774,2024-06-30,2020-09-17,NaN,annuel,naq,42,telephone,52.09,1.8,1,0,NaN,52.09,1,0


## 3 — Un client est-il une ligne unique ?

Compter les snapshots par client et auditer les trajectoires. Des clients répétés sont attendus ; plusieurs churns ou des lignes après churn demandent une explication.

In [3]:
counts = data.groupby("customer_id").size()
ordered = data.sort_values(["customer_id", "snapshot_date"]).copy()
ordered["prior_churn"] = ordered.groupby("customer_id").churn.transform(
    lambda x: x.fillna(0).shift(fill_value=0).cummax()
)
client_labels = data.groupby("customer_id").churn.agg(observed="count", positive="sum")
structure = pd.Series(
    {
        "snapshots": len(data),
        "clients": len(counts),
        "clients_repeated": int(counts.gt(1).sum()),
        "clients_with_observed_churn": int(client_labels.positive.gt(0).sum()),
        "clients_without_label": int(client_labels.observed.eq(0).sum()),
        "clients_with_multiple_churns": int(client_labels.positive.gt(1).sum()),
        "snapshots_after_churn": int(ordered.prior_churn.eq(1).sum()),
    },
    name="count",
)
display(structure.to_frame())
distribution = counts.value_counts().sort_index()
fig, ax = plt.subplots(figsize=(FIG_WIDTH, PANEL_HEIGHT))
ax.bar(distribution.index.astype(str), distribution, color="#2563EB")
ax.set(title="Longueur des trajectoires observées", xlabel="Snapshots par client", ylabel="Clients")
show_figure(fig, "client_history")
print(
    f"Prévalence par snapshot : {labeled.churn.mean():.2%} ; clients avec churn observé : {client_labels.positive.gt(0).mean():.2%}."
)

,count
snapshots,38629
clients,4000
clients_repeated,3844
clients_with_observed_churn,1162
clients_without_label,0
clients_with_multiple_churns,0
snapshots_after_churn,0


Prévalence par snapshot : 3.01% ; clients avec churn observé : 29.05%.


## 4 — Comment les clients se répartissent-ils entre les mois ?

Première apparition dans le fichier et date de souscription sont différentes. Les sorties du panel ne prouvent pas à elles seules une résiliation.

In [4]:
panel = data.assign(month=data.snapshot_date.dt.to_period("M"))
months = pd.period_range(panel.month.min(), panel.month.max(), freq="M")
first = panel.groupby("customer_id").month.min()
monthly = (
    panel.groupby("month")
    .agg(snapshots=("customer_id", "size"), clients=("customer_id", "nunique"))
    .reindex(months, fill_value=0)
)
monthly["first_appearance"] = first.value_counts().reindex(months, fill_value=0)
display(monthly)
cohorts = pd.crosstab(panel.customer_id.map(first), panel.month).reindex(
    columns=months, fill_value=0
)
fig, axes = plt.subplots(2, 1, figsize=(FIG_WIDTH, 8))
axes[0].plot(monthly.index.astype(str), monthly.clients, "o-", label="Clients")
axes[0].bar(
    monthly.index.astype(str), monthly.first_appearance, alpha=0.4, label="Premières apparitions"
)
axes[0].set(title="Volumes mensuels", ylabel="Clients")
axes[0].tick_params(axis="x", rotation=30)
axes[0].legend()
im = axes[1].imshow(cohorts.to_numpy(), cmap="Blues", aspect="auto")
axes[1].set_xticks(range(len(months)), months.astype(str), rotation=35)
axes[1].set_yticks(range(len(cohorts)), cohorts.index.astype(str))
axes[1].set(title="Cohorte de première apparition × mois observé", ylabel="Cohorte")
fig.colorbar(im, ax=axes[1], label="Clients observés")
show_figure(fig, "monthly_panel")

,snapshots,clients,first_appearance
2024-01,3488,3488,3488
2024-02,3422,3422,45
2024-03,3360,3360,46
2024-04,3299,3299,45
2024-05,3284,3284,70
2024-06,3250,3250,68
2024-07,3219,3219,47
2024-08,3183,3183,77
2024-09,3126,3126,56
2024-10,3088,3088,58


## 5 — Contrats, canaux et géographie

Les distributions par snapshot et par premier contrat client ont des dénominateurs différents. Les comptes clients par catégorie peuvent se chevaucher si le client change de catégorie.

In [5]:
first_contract = (
    data.sort_values(["customer_id", "snapshot_date"]).drop_duplicates("customer_id").contract
)
contract_profile = pd.concat(
    [
        data.contract.value_counts(normalize=True).rename("share_snapshots"),
        first_contract.value_counts(normalize=True).rename("share_first_contract"),
    ],
    axis=1,
)
display(contract_profile)
print(
    "Clients avec changement de contrat observé :",
    int(data.groupby("customer_id").contract.nunique().gt(1).sum()),
)
for column in ["contract", "channel", "region", "dept"]:
    temp = data.assign(category=data[column].fillna("Non renseigné"))
    table = (
        temp.groupby("category")
        .agg(snapshots=("customer_id", "size"), clients=("customer_id", "nunique"))
        .sort_values("snapshots", ascending=False)
    )
    display(table)
    fig, ax = plt.subplots(figsize=(FIG_WIDTH, max(4, len(table.head(15)) * 0.3)))
    ax.barh(table.head(15).index, table.head(15).snapshots, color="#2563EB")
    ax.invert_yaxis()
    ax.set(title=f"Répartition de {column} — 15 modalités maximum", xlabel="Snapshots")
    show_figure(fig, f"composition_{column}")

,share_snapshots,share_first_contract
contract,,
mensuel,0.516917,0.56275
annuel,0.323902,0.295
biannuel,0.159181,0.14225


Clients avec changement de contrat observé : 0


,snapshots,clients
category,,
mensuel,19968,2251
annuel,12512,1180
biannuel,6149,569


,snapshots,clients
category,,
web,15332,1586
boutique,11507,1195
telephone,8006,825
partenaire,3784,394


,snapshots,clients
category,,
ara,5800,587
naq,5639,573
idf,5588,606
occ,5490,561
bre,5447,564
ges,5357,557
paca,5308,552


,snapshots,clients
category,,
01,5306,563
02,3346,342
03,2333,239
04,1795,176
05,1695,168
06,1360,139
07,1295,123
10,942,96
09,932,94


### 5.1 — Quelles informations changent chez un même client ?

Comparer les variations entre mois consécutifs et la disponibilité des mesures. Une mensualité différente n'est pas automatiquement une hausse commerciale : les évolutions du fichier doivent être expliquées avant d'interpréter la feature de prix. Une note absente n'est pas une mauvaise note.


In [6]:
trajectories = data.sort_values(["customer_id", "snapshot_date"]).copy()
month_number = trajectories.snapshot_date.dt.year * 12 + trajectories.snapshot_date.dt.month
consecutive = month_number.sub(month_number.groupby(trajectories.customer_id).shift()).eq(1)
variation_rows = []
for column in ["monthly_fee", "data_gb", "support_calls_3m", "late_payments", "nps"]:
    values = trajectories[column]
    previous = values.groupby(trajectories.customer_id).shift()
    valid = consecutive & values.notna() & previous.notna()
    changed = valid & values.ne(previous)
    variation_rows.append(
        {
            "column": column,
            "comparable_pairs": int(valid.sum()),
            "changed_pairs": int(changed.sum()),
            "changed_pct": changed.sum() / valid.sum() * 100 if valid.any() else np.nan,
        }
    )
display(pd.DataFrame(variation_rows))
missing_columns = ["monthly_fee", "nps"]
monthly_missing = data.groupby(data.snapshot_date.dt.to_period("M").astype(str))[
    missing_columns
].agg(lambda values: values.isna().mean() * 100)
fig, ax = plt.subplots(figsize=(FIG_WIDTH, PANEL_HEIGHT))
monthly_missing.plot(ax=ax, marker="o")
ax.set(
    title="Disponibilité des prix et du NPS selon les mois", xlabel="Mois", ylabel="Manquants (%)"
)
ax.tick_params(axis="x", rotation=30)
show_figure(fig, "availability_over_time")

,column,comparable_pairs,changed_pairs,changed_pct
0,monthly_fee,31138,31095,99.861905
1,data_gb,34629,34237,98.868001
2,support_calls_3m,34629,23874,68.942216
3,late_payments,34629,16098,46.487048
4,nps,3747,2976,79.423539


## 6 — Distributions et relations entre colonnes

Spearman mesure les associations monotones par snapshot. Il ne démontre pas de causalité ; les observations répétées ne sont pas indépendantes. Vérifier les unités de consommation avant de traiter ses extrêmes.

In [7]:
numeric = [
    "monthly_fee",
    "data_gb",
    "support_calls_3m",
    "late_payments",
    "nps",
    "last_invoice_amount",
]
numeric = [c for c in numeric if c in data]
display(data[numeric].describe(percentiles=[0.01, 0.5, 0.99]).T)
fig, axes = plt.subplots(len(numeric), 1, figsize=(FIG_WIDTH, 2.2 * len(numeric)))
for ax, column in zip(axes, numeric):
    ax.hist(data[column].dropna(), bins=40, color="#2563EB")
    ax.set(title=column, ylabel="Snapshots")
show_figure(fig, "numeric_distributions")
corr = data[numeric].corr(method="spearman", min_periods=30)
available = data[numeric].notna().astype(int)
display(corr, available.T @ available)
fig, ax = plt.subplots(figsize=(FIG_WIDTH, 6))
im = ax.imshow(corr.to_numpy(), cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(numeric)), numeric, rotation=35, ha="right")
ax.set_yticks(range(len(numeric)), numeric)
for i in range(len(numeric)):
    for j in range(len(numeric)):
        if pd.notna(corr.iloc[i, j]):
            ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center")
fig.colorbar(im, ax=ax, label="Corrélation de rang")
show_figure(fig, "correlation")

,count,mean,std,min,1%,50%,99%,max
monthly_fee,36620.0,60.962549,18.110659,10.34,18.8638,60.86,103.5781,130.39
data_gb,38629.0,71.908701,767.045733,0.00,0.2000,3.70,1946.0000,32154.00
support_calls_3m,38629.0,1.142302,1.163153,0.00,0.0000,1.00,5.0000,10.00
late_payments,38629.0,0.400761,0.643364,0.00,0.0000,0.00,2.0000,5.00
nps,12427.0,6.803895,1.979882,-0.00,2.0000,7.00,10.0000,10.00
last_invoice_amount,38629.0,59.982277,18.719116,2.41,15.3228,60.18,103.2388,130.39


,monthly_fee,data_gb,support_calls_3m,late_payments,nps,last_invoice_amount
monthly_fee,1.000000,-0.004798,-0.003205,0.002434,0.011539,0.957970
data_gb,-0.004798,1.000000,-0.002479,0.002978,-0.017308,0.001069
support_calls_3m,-0.003205,-0.002479,1.000000,0.058538,-0.236119,-0.025700
late_payments,0.002434,0.002978,0.058538,1.000000,-0.106769,-0.015418
nps,0.011539,-0.017308,-0.236119,-0.106769,1.000000,0.021921
last_invoice_amount,0.957970,0.001069,-0.025700,-0.015418,0.021921,1.000000


,monthly_fee,data_gb,support_calls_3m,late_payments,nps,last_invoice_amount
monthly_fee,36620,36620,36620,36620,11766,36620
data_gb,36620,38629,38629,38629,12427,38629
support_calls_3m,36620,38629,38629,38629,12427,38629
late_payments,36620,38629,38629,38629,12427,38629
nps,11766,12427,12427,12427,12427,12427
last_invoice_amount,36620,38629,38629,38629,12427,38629


## 7 — Vérifier les signaux suspects de facture

Comparer la facture à la mensualité ne valide pas leur disponibilité avant la résiliation. Cette famille est exclue du modèle de référence de la refonte.

In [8]:
if "last_invoice_amount" in data:
    relation = pd.Series("Non évaluable", index=data.index)
    valid = data.monthly_fee.notna() & data.last_invoice_amount.notna()
    relation.loc[valid & data.last_invoice_amount.eq(data.monthly_fee)] = "Facture = mensualité"
    relation.loc[valid & data.last_invoice_amount.lt(data.monthly_fee)] = "Facture < mensualité"
    relation.loc[valid & data.last_invoice_amount.gt(data.monthly_fee)] = "Facture > mensualité"
    display(pd.crosstab(relation, data.churn))

churn,0,1
row_0,,
Facture < mensualité,0,1098
Facture = mensualité,35522,0
Non évaluable,1945,64


## 8 — Helpers des dix KPI

Chaque taux utilise les snapshots étiquetés. Les tableaux donnent aussi les effectifs, positifs et clients distincts ; les petits groupes ne prouvent pas une différence stable.

In [9]:
eda = data.copy()
eda["tenure_months"] = (eda.snapshot_date - eda.signup_date).dt.days / 30.4375
eda["tenure_months"] = eda.tenure_months.where(eda.tenure_months.ge(0))
global_rate = labeled.churn.mean() * 100


def summarize(frame, group):
    return (
        frame.loc[frame.churn.notna()]
        .groupby(group, dropna=False, observed=True)
        .agg(
            snapshots=("churn", "size"),
            positives=("churn", "sum"),
            clients=("customer_id", "nunique"),
            rate=("churn", "mean"),
        )
        .reset_index()
    )


def plot_rate(table, group, title, name):
    labels = table[group].astype("string").fillna("Non renseigné")
    fig, ax = plt.subplots(figsize=(FIG_WIDTH, PANEL_HEIGHT))
    bars = ax.bar(labels, table.rate * 100, color="#2563EB")
    ax.bar_label(bars, fmt="%.2f %%", padding=3)
    ax.axhline(global_rate, color="#E11D48", linestyle="--", label="Prévalence globale")
    ax.set(title=title, ylabel="Churn par snapshot (%)")
    ax.tick_params(axis="x", rotation=25)
    ax.margins(y=0.25)
    ax.legend()
    show_figure(fig, name)
    display(table)

### KPI 1 — Prévalence

Quelle part des snapshots est positive ? Ne pas confondre ce taux avec la proportion de clients ayant un événement sur l’année.

In [10]:
dist = labeled.groupby("churn").size()
fig, ax = plt.subplots(figsize=(FIG_WIDTH, PANEL_HEIGHT))
ax.bar(["Non-churn", "Churn"], dist.reindex([0, 1], fill_value=0), color=["#059669", "#E11D48"])
ax.set(title=f"Prévalence par snapshot : {global_rate:.2f} %", ylabel="Snapshots")
show_figure(fig, "kpi_01")
display(dist.to_frame("snapshots"))

,snapshots
churn,
0,37467
1,1162


### KPI 2 — Contrat

L’engagement est-il associé au churn ? Les différences peuvent refléter la composition des groupes.

In [11]:
plot_rate(summarize(eda, "contract"), "contract", "Churn par contrat", "kpi_02")

,contract,snapshots,positives,clients,rate
0,annuel,12512,155,1180,0.012388
1,biannuel,6149,43,569,0.006993
2,mensuel,19968,964,2251,0.048277


### KPI 3 — Ancienneté

Le taux varie-t-il avec le cycle de vie ? Les tranches servent ici à décrire, sans sélectionner un modèle.

In [12]:
temp = eda.copy()
temp["tenure_group"] = pd.cut(
    temp.tenure_months,
    [0, 3, 6, 12, 24, 48, np.inf],
    right=False,
    labels=["0–3", "3–6", "6–12", "12–24", "24–48", "48+"],
)
plot_rate(summarize(temp, "tenure_group"), "tenure_group", "Churn et ancienneté (mois)", "kpi_03")

,tenure_group,snapshots,positives,clients,rate
0,0–3,1754,130,693,0.074116
1,3–6,1632,78,682,0.047794
2,6–12,3377,116,844,0.034350
3,12–24,6654,206,1166,0.030959
4,24–48,14096,378,1876,0.026816
5,48+,10943,251,1295,0.022937
6,NaN,173,3,169,0.017341


### KPI 4 — support_calls_3m

Le recours au support est-il associé au churn ? Le compteur couvre trois mois.

In [13]:
temp = eda.copy()
temp["group"] = (
    temp["support_calls_3m"]
    .clip(upper=3)
    .map({0: "0", 1: "1", 2: "2", 3: "3+"})
    .fillna("Non renseigné")
)
plot_rate(summarize(temp, "group"), "group", "Churn selon support_calls_3m", "kpi_04")

,group,snapshots,positives,clients,rate
0,0,13500,200,3457,0.014815
1,1,13084,337,3640,0.025757
2,2,7389,276,3174,0.037353
3,3+,4656,349,2403,0.074957


### KPI 5 — late_payments

Les retards sont-ils associés au churn ? La période de ce compteur reste à documenter.

In [14]:
temp = eda.copy()
temp["group"] = (
    temp["late_payments"]
    .clip(upper=3)
    .map({0: "0", 1: "1", 2: "2", 3: "3+"})
    .fillna("Non renseigné")
)
plot_rate(summarize(temp, "group"), "group", "Churn selon late_payments", "kpi_05")

,group,snapshots,positives,clients,rate
0,0,26038,590,3885,0.022659
1,1,10122,393,3564,0.038826
2,2,2091,134,1574,0.064084
3,3+,378,45,356,0.119048


### KPI 6 — Satisfaction

Les notes faibles sont-elles associées au churn ? L’absence de note ne signifie pas insatisfaction.

In [15]:
temp = eda.copy()
temp["nps_group"] = (
    pd.cut(temp.nps, [-0.1, 6, 8, 10], labels=["0–6", ">6–8", ">8–10"])
    .astype("string")
    .fillna("Non renseigné")
)
plot_rate(summarize(temp, "nps_group"), "nps_group", "Churn et satisfaction", "kpi_06")

,nps_group,snapshots,positives,clients,rate
0,0–6,5413,228,2320,0.042121
1,>6–8,4332,130,2467,0.030009
2,>8–10,2682,51,1629,0.019016
3,Non renseigné,26202,753,3911,0.028738


### KPI 7 — Tranches tarifaires

Visualiser le découpage du 01 : bornes sur janvier–juillet, appliquées ensuite sans recalcul. Ce regroupement ne démontre pas une sensibilité causale au prix.

In [16]:
reference = (
    eda["monthly_fee"]
    .loc[eda.snapshot_date.dt.to_period("M").between("2024-01", "2024-07")]
    .dropna()
)
cuts = np.unique(reference.quantile([1 / 3, 2 / 3]).to_numpy())
cuts = cuts[(cuts > reference.min()) & (cuts < reference.max())]
labels = {
    1: ["Intermédiaire"],
    2: ["Peu coûteux", "Coûteux"],
    3: ["Peu coûteux", "Intermédiaire", "Coûteux"],
}[len(cuts) + 1]
temp = eda.copy()
temp["fee_band"] = (
    pd.Series(
        pd.cut(temp.monthly_fee, np.r_[-np.inf, cuts, np.inf], labels=labels), index=temp.index
    )
    .astype("string")
    .fillna("Non renseigné")
)
fig, ax = plt.subplots(figsize=(FIG_WIDTH, PANEL_HEIGHT))
ax.hist(reference, bins=40, color="#2563EB", alpha=0.7)
for cut in cuts:
    ax.axvline(cut, color="#E11D48", linestyle="--", label=f"Borne : {cut:.2f} €")
ax.set(title="Découpage tarifaire du train", xlabel="Mensualité (€)", ylabel="Snapshots")
ax.legend()
show_figure(fig, "fee_band_distribution")
plot_rate(
    summarize(temp, "fee_band"),
    "fee_band",
    "Churn par tranche tarifaire — descriptif annuel",
    "kpi_07",
)

,fee_band,snapshots,positives,clients,rate
0,Coûteux,12554,440,1662,0.035049
1,Intermédiaire,12164,344,1802,0.028280
2,Non renseigné,2009,64,1571,0.031857
3,Peu coûteux,11902,314,1519,0.026382


### KPI 8 — Consommation

Le risque varie-t-il selon l’usage ? Les quintiles annuels sont descriptifs, pas des bornes de modélisation.

In [17]:
temp = eda.copy()
temp["usage_group"] = (
    pd.qcut(temp.data_gb, 5, duplicates="drop").astype("string").fillna("Non renseigné")
)
plot_rate(summarize(temp, "usage_group"), "usage_group", "Churn et consommation", "kpi_08")

,usage_group,snapshots,positives,clients,rate
0,"(-0.001, 1.6]",8373,339,2938,0.040487
1,"(1.6, 2.9]",7493,284,3143,0.037902
2,"(2.9, 4.7]",7484,171,3133,0.022849
3,"(4.7, 7.9]",7690,185,3023,0.024057
4,"(7.9, 32154.0]",7589,183,2592,0.024114


### KPI 9 — Contrat et ancienneté

Comparer des groupes suffisamment peuplés. Les taux des groupes de moins de 100 snapshots sont masqués, leurs effectifs restent visibles.

In [18]:
temp = eda.copy()
temp["tenure_group"] = pd.cut(
    temp.tenure_months,
    [0, 3, 6, 12, 24, 48, np.inf],
    right=False,
    labels=["0–3", "3–6", "6–12", "12–24", "24–48", "48+"],
)
stats = (
    temp.loc[temp.churn.notna()]
    .groupby(["tenure_group", "contract"], observed=True)
    .agg(snapshots=("churn", "size"), rate=("churn", "mean"))
    .reset_index()
)
stats["display_rate"] = (stats.rate * 100).where(stats.snapshots.ge(100))
table = stats.pivot(index="tenure_group", columns="contract", values="display_rate")
fig, ax = plt.subplots(figsize=(FIG_WIDTH, 5))
im = ax.imshow(np.ma.masked_invalid(table.to_numpy()), aspect="auto", cmap="Reds")
ax.set_xticks(range(len(table.columns)), table.columns)
ax.set_yticks(range(len(table)), table.index.astype(str))
ax.set(title="Contrat × ancienneté", ylabel="Ancienneté (mois)")
fig.colorbar(im, ax=ax, label="Churn (%)")
show_figure(fig, "kpi_09")
display(stats)

,tenure_group,contract,snapshots,rate,display_rate
0,0–3,annuel,556,0.023381,2.338129
1,0–3,biannuel,306,0.019608,1.960784
2,0–3,mensuel,892,0.124439,12.443946
3,3–6,annuel,560,0.023214,2.321429
4,3–6,biannuel,289,0.003460,0.346021
5,3–6,mensuel,783,0.081737,8.173691
6,6–12,annuel,1074,0.018622,1.862197
7,6–12,biannuel,547,0.003656,0.365631
8,6–12,mensuel,1756,0.053531,5.353075
9,12–24,annuel,2022,0.014342,1.434224


### KPI 10 — Temps

Les variations de taux peuvent refléter une composition changeante du panel. Le mois du snapshot n’est pas le mois exact de l’événement.

In [19]:
temp = eda.assign(month=eda.snapshot_date.dt.to_period("M").astype(str))
plot_rate(summarize(temp, "month"), "month", "Prévalence mensuelle", "kpi_10")

,month,snapshots,positives,clients,rate
0,2024-01,3488,111,3488,0.031823
1,2024-02,3422,108,3422,0.031560
2,2024-03,3360,106,3360,0.031548
3,2024-04,3299,85,3299,0.025765
4,2024-05,3284,102,3284,0.031060
5,2024-06,3250,78,3250,0.024000
6,2024-07,3219,113,3219,0.035104
7,2024-08,3183,113,3183,0.035501
8,2024-09,3126,96,3126,0.030710
9,2024-10,3088,85,3088,0.027526


## 9 — Questions à résoudre avant de conclure

- Les unités et périodes de consommation sont-elles homogènes ?
- Pourquoi les tarifs changent-ils presque chaque mois ?
- Quand les notes de satisfaction et les factures deviennent-elles disponibles ?
- Quels groupes ont assez de clients distincts pour une interprétation stable ?

Les variables de la référence sont définies dans le 01. Leurs hypothèses se testent sur validation temporelle dans le 02 ; l'explicabilité du 03 ne prouve pas de causalité.